### Currency convertor Agent 

In [57]:
from langchain_ollama import ChatOllama

llm = ChatOllama(model = 'ministral-3:3b')

In [58]:
from langchain_core.tools import tool
import requests

@tool 
def currency_convertor(amount: int, from_currency: str, to: str) -> dict:
    '''convert currency by using ISO 4217 Three Letter Currency Codes - e.g. USD for US Dollars, EUR for Euros, JPY for Japanese Yen etc'''

    result = requests.get(f"https://v6.exchangerate-api.com/v6/{'a3a9b4d36225c6a18ae6ff54'}/latest/{from_currency}")

    # print(result.json()["conversion_rates"][to] * amount)

    return result.json()["conversion_rates"][to] * amount

    

In [59]:
currency_convertor.invoke({"amount": 100, "from_currency": "USD", "to": "INR"})

9095.89

In [60]:
llm_with_tool = llm.bind_tools([currency_convertor])

In [61]:
from langchain_core.messages import HumanMessage


messages = []

query = HumanMessage(content="hi, can you convert 100 indian rupees to european Euro currency for me?")

messages.append(query)


In [62]:
# Tool Calling LLM 

tool_calling = llm_with_tool.invoke(messages)

In [63]:
tool_calling

AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'ministral-3:3b', 'created_at': '2026-01-20T21:02:39.9308823Z', 'done': True, 'done_reason': 'stop', 'total_duration': 2178720600, 'load_duration': 444373000, 'prompt_eval_count': 672, 'prompt_eval_duration': 742933600, 'eval_count': 27, 'eval_duration': 969421100, 'logprobs': None, 'model_name': 'ministral-3:3b', 'model_provider': 'ollama'}, id='lc_run--019bdd37-74b2-7dc2-bece-7b43726be6cd-0', tool_calls=[{'name': 'currency_convertor', 'args': {'amount': 100, 'from_currency': 'INR', 'to': 'EUR'}, 'id': 'cd49bcd6-e379-4eaf-be56-2d75e28df11d', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 672, 'output_tokens': 27, 'total_tokens': 699})

In [64]:
messages.append(tool_calling)

In [65]:
tool_calling.tool_calls[0]

{'name': 'currency_convertor',
 'args': {'amount': 100, 'from_currency': 'INR', 'to': 'EUR'},
 'id': 'cd49bcd6-e379-4eaf-be56-2d75e28df11d',
 'type': 'tool_call'}

In [66]:
tool_execution_result  = currency_convertor.invoke(tool_calling.tool_calls[0])

In [67]:
tool_execution_result

ToolMessage(content='0.9456000000000001', name='currency_convertor', tool_call_id='cd49bcd6-e379-4eaf-be56-2d75e28df11d')

In [68]:
messages.append(tool_execution_result)

In [69]:
# now finally our messages list looks like this 
messages

[HumanMessage(content='hi, can you convert 100 indian rupees to european Euro currency for me?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'ministral-3:3b', 'created_at': '2026-01-20T21:02:39.9308823Z', 'done': True, 'done_reason': 'stop', 'total_duration': 2178720600, 'load_duration': 444373000, 'prompt_eval_count': 672, 'prompt_eval_duration': 742933600, 'eval_count': 27, 'eval_duration': 969421100, 'logprobs': None, 'model_name': 'ministral-3:3b', 'model_provider': 'ollama'}, id='lc_run--019bdd37-74b2-7dc2-bece-7b43726be6cd-0', tool_calls=[{'name': 'currency_convertor', 'args': {'amount': 100, 'from_currency': 'INR', 'to': 'EUR'}, 'id': 'cd49bcd6-e379-4eaf-be56-2d75e28df11d', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 672, 'output_tokens': 27, 'total_tokens': 699}),
 ToolMessage(content='0.9456000000000001', name='currency_convertor', tool_call_id='cd49bcd6-e379-4eaf-be56-2

In [70]:
final_response = llm_with_tool.invoke(messages)

In [71]:
final_response

AIMessage(content='100 Indian Rupees (INR) is approximately **10.00 Euro (EUR)**.\n\n*(Note: The exact conversion rate may vary slightly depending on the current exchange rate.)* Would you like any further assistance?', additional_kwargs={}, response_metadata={'model': 'ministral-3:3b', 'created_at': '2026-01-20T21:02:42.8415185Z', 'done': True, 'done_reason': 'stop', 'total_duration': 2436532100, 'load_duration': 315430300, 'prompt_eval_count': 717, 'prompt_eval_duration': 294782900, 'eval_count': 51, 'eval_duration': 1784036100, 'logprobs': None, 'model_name': 'ministral-3:3b', 'model_provider': 'ollama'}, id='lc_run--019bdd37-7f13-7c20-b555-f3b87b27ae1c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 717, 'output_tokens': 51, 'total_tokens': 768})